In [ ]:
%pip install openai gradio python-dotenv requests 

In [ ]:
#Imports and enviroment setup
import os
import json
import re
import xml.etree.ElementTree as ET
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# OpenRouter client (OpenAI-compatible)
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# Ollama local client
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print("Clients ready ✅")

In [ ]:
#Fetch latest news
def fetch_latest_news(topic: str):

    url = f"https://news.google.com/rss/search?q={topic}"
    response = requests.get(url)
    root = ET.fromstring(response.content)

    # Find items (works with or without XML namespaces)
    items = root.findall(".//item") or root.findall(".//{*}item")

    headlines = []
    for item in items[:5]:
        title = item.find("title")
        if title is None:
            title = item.find("{*}title")
        if title is not None and title.text:
            headlines.append(title.text)

    if not headlines:
        return {"error": "No news found"}

    return {"headlines": headlines}

In [ ]:
#tools schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "fetch_latest_news",
            "description": "Fetch latest news headlines for a given topic using Google News RSS.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {
                        "type": "string",
                        "description": "The topic to search news for"
                    }
                },
                "required": ["topic"]
            }
        }
    }
]

In [ ]:
def _parse_tool_args(raw_args: str, user_message: str) -> dict:
    """Parse tool arguments robustly; Ollama often returns malformed JSON."""
    topic = None
    try:
        parsed = json.loads(raw_args)
        if isinstance(parsed, dict):
            topic = parsed.get("topic") or parsed.get("Topic")
            if isinstance(topic, list):
                topic = topic[0] if topic else None
    except json.JSONDecodeError:
        pass
    if topic is None or not str(topic).strip():
        match = re.search(r'"topic"\s*:\s*"([^"]+)"', raw_args)
        if match:
            topic = match.group(1).strip()
    if topic is None or not str(topic).strip():
        topic = user_message.strip() or "general news"
    return {"topic": str(topic)}

def chat(message, history, model_choice, summary_style):

    # Select client + model
    if model_choice == "OpenRouter":
        client = openrouter_client
        model = "openai/gpt-4o-mini"
    else:
        client = ollama_client
        model = "llama3.2:1b"

    system_prompt = f"""
    You are a professional AI News Intelligence Assistant.
    Always call the tool when the user asks about current or recent news.
    After receiving headlines, summarize them in the following style: {summary_style}.
    Be clear and structured.
    """

    history = [{"role": h["role"], "content": h["content"]} for h in (history or [])]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools
    )

    msg = response.choices[0].message

    # Tool call handling
    if msg.tool_calls:

        tool_call = msg.tool_calls[0]
        raw_args = tool_call.function.arguments or "{}"
        args = _parse_tool_args(raw_args, message)
        tool_result = fetch_latest_news(**args)

        messages.append(msg)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(tool_result)
        })

        final_response = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )

        partial = ""
        for chunk in final_response:
            if chunk.choices and chunk.choices[0].delta.content:
                partial += chunk.choices[0].delta.content
                yield partial

    else:
        yield msg.content

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("## 📰 AI News Intelligence Assistant")

    with gr.Row():
        model_selector = gr.Dropdown(
            ["OpenRouter", "Ollama"],
            value="OpenRouter",
            label="Model Provider"
        )

        summary_selector = gr.Dropdown(
            ["Bullet Points", "Executive Summary", "Simple Explanation"],
            value="Bullet Points",
            label="Summary Style"
        )

    with gr.Row():
        msg = gr.Textbox(label="Ask about latest news...", show_label=False, container=False, placeholder="Ask about latest news...")
        submit_btn = gr.Button("Submit", variant="primary")
    chatbot = gr.Chatbot(type="messages", render_markdown=True)

    def user_submit(user_message, history):
        history = history or []
        return "", history + [{"role": "user", "content": user_message}]

    def bot_response(history, model_choice, summary_style):
        user_message = history[-1]["content"]

        for partial in chat(user_message, history[:-1], model_choice, summary_style):
            yield history + [{"role": "assistant", "content": partial}]

    submit_event = msg.submit(user_submit, [msg, chatbot], [msg, chatbot])
    submit_event = submit_event.then(bot_response, [chatbot, model_selector, summary_selector], chatbot)
    submit_btn.click(user_submit, [msg, chatbot], [msg, chatbot]).then(
        bot_response,
        [chatbot, model_selector, summary_selector],
        chatbot
    )

demo.launch()